In [ ]:
from pathlib import Path
import pandas as pd
import hvplot.pandas

import holoviews as hv
from workspace_dani.data_preprocessing.utils import data_handling
from workspace_dani.visualizations.donut_plots import make_donut_plot

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import palettable

%matplotlib inline

hv.extension("bokeh", "matplotlib")


In [ ]:
lab_lut_path = Path.cwd().parents[3] / Path("intermediary_data/look_up_tables/dhm_eicu_lab_look_up_table_valid.parquet")

In [ ]:
df_lut = data_handling.load_parquet_table(parquet_path=lab_lut_path, backend="numpy_nullable", debug_mode=False,
                                          verbose=True)

In [ ]:
df_lut.describe(include="all")

303 

In [ ]:
df_labs = data_handling.load_parquet_table(
    Path.cwd().parents[3] / Path("intermediary_data/tables_basis/2024-03-26/lab_regular.parquet"),
    backend="numpy_nullable")

In [ ]:
df_labs.head()

### Checking if our our lab_eicu names actually correspond with the names of the lab table

In [ ]:
available_labs = df_labs["labname"].unique().tolist()

In [ ]:
available_labs_dict = dict(zip(available_labs, range(len(available_labs))))

In [ ]:
for lab in df_lut["lab_eicu"]:
    try:
        available_labs_dict[lab]
    except KeyError as ex:
        print(ex)

In [ ]:
matched_lab_names = df_lut["lab_eicu"].unique()

-> labname and lab_eicu appear to be consistent!

In [ ]:
df_labs_matched = df_labs[df_labs["labname"].isin(matched_lab_names)]

In [ ]:
cat_uniq = df_labs_matched["labname"].unique()
type(cat_uniq)

In [ ]:
# df_labs_no_match = df_labs[~df_labs["labname"].isin(df_lut["lab_eicu"])]
df_labs_no_match = df_labs[~df_labs["labname"].isin(matched_lab_names)]

In [ ]:
len(df_labs) - len(df_labs_no_match) - len(df_labs_matched)

In [ ]:
labs_matched_freq: dict = df_labs_matched["labname"].value_counts().to_dict()
len(labs_matched_freq)

In [ ]:
labs_no_match_freq: dict = df_labs_no_match["labname"].value_counts().to_dict()

In [ ]:
df_labs_matched_freq = pd.Series(labs_matched_freq).sort_values(ascending=False).reset_index()
df_labs_no_match_freq = pd.Series(labs_no_match_freq).sort_values(ascending=False).reset_index()

df_labs_matched_freq.columns = ["labname", "count"]
df_labs_no_match_freq.columns = ["labname", "count"]

df_labs_matched_freq["match"] = df_labs_matched_freq.apply(lambda x: "yes", axis=1)
df_labs_no_match_freq["match"] = df_labs_no_match_freq.apply(lambda x: "no", axis=1)

In [ ]:
df_labs_matched_freq = df_labs_matched_freq[df_labs_matched_freq["count"] > 0]
df_labs_no_match_freq = df_labs_no_match_freq[df_labs_no_match_freq["count"] > 0]


In [ ]:
df_labs_freq = pd.concat([df_labs_matched_freq, df_labs_no_match_freq], axis=0)

In [ ]:
from holoviews.plotting.util import process_cmap
from holoviews import opts

cmap_list = hv.plotting.list_cmaps(category="Uniform Sequential", provider='bokeh')

In [ ]:
hv.Cycle.default_cycles.keys()
color_cycle = hv.Cycle("Colorblind")
hv.opts.defaults(opts.Bars(color=color_cycle))


In [ ]:
match_freq_plot = df_labs_matched_freq.hvplot.bar(x="labname", y="count", label="match", min_height=720, min_width=960,
                                                  responsive=True, logy=True)

In [ ]:
no_match_freq_plot = df_labs_no_match_freq.hvplot.bar(x="labname", y="count", label="no match", min_height=720,
                                                      min_width=960,
                                                      responsive=True, logy=True)

In [ ]:
match_plot = match_freq_plot * no_match_freq_plot

In [ ]:
match_plot.opts(shared_axes=True, fontscale=1, xrotation=60, title="Lab Test Counts")

In [ ]:
df_patients: pd.DataFrame = data_handling.load_parquet_table(
    (Path.cwd().parents[3] / Path("intermediary_data/dataset/build/2024-03-26/build_dataset_cats.parquet")))

### Investigating the distribution of lab tests across ICU unit types

In [ ]:
df_patients[df_patients["patientunitstayid"] == 141168]["unittype"]

In [ ]:
df_labs_matched["patientunitstayid"]

In [ ]:
frame = df_patients.loc[df_patients.patientunitstayid == 141168, "unittype"]

In [ ]:
from tqdm.auto import tqdm

tqdm.pandas()

In [ ]:
def get_unittype_via_ids(x):
    icu_type = df_patients.loc[df_patients.patientunitstayid == x, "unittype"]
    return icu_type


In [ ]:
# ids_unittyp = df_labs_matched["patientunitstayid"].progress_apply(func=get_unittype_via_ids)
df_labs.head()

In [ ]:
df_patients.head()

In [ ]:
import polars as pl

pl_df_labs: pl.DataFrame = pl.from_pandas(df_labs)
pl_df_patients: pl.DataFrame = pl.from_pandas(df_patients)

In [ ]:
pl_df_patients_icu_type = pl_df_patients.select(pl.col("patientunitstayid", "unittype"))

In [ ]:
pl_df_labs_icu_type = pl_df_labs.join(pl_df_patients_icu_type, on="patientunitstayid", how="left")

In [ ]:
df_labs_icu_type = pl_df_labs_icu_type.to_pandas()

In [ ]:
labs_icu_type_freq = df_labs_icu_type.value_counts(subset=["labname", "unittype"]).reset_index()

In [ ]:
labs_icu_type_freq["match"] = labs_icu_type_freq["labname"].apply(lambda x: "yes" if x in matched_lab_names else "no")

In [ ]:
match_icu_freq_plot = labs_icu_type_freq[labs_icu_type_freq["match"] == "yes"].hvplot.bar(x="labname", y="count",
                                                                                          by="unittype", stacked=True,
                                                                                          label="match").opts(
    xrotation=60, responsive=True, min_height=720, min_width=960)
# responsive=True)
no_match_icu_freq_plot = labs_icu_type_freq[labs_icu_type_freq["match"] == "no"].hvplot.bar(x="labname", y="count",
                                                                                            by="unittype", stacked=True,
                                                                                            label="no match").opts(
    xrotation=60, responsive=True, min_height=720, min_width=960)
labs_icu_plot = match_icu_freq_plot + no_match_icu_freq_plot

In [ ]:
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
labs_icu_plot.opts(shared_axes=False, fontscale=1, title="All Lab Test Counts per ICU Type", )                  

### Same plot with percentages

In [ ]:
labs_icu_type_freq_perc = df_labs_icu_type.value_counts(subset=["labname", "unittype"], normalize=True).reset_index()
labs_icu_type_freq_perc["match"] = labs_icu_type_freq_perc["labname"].apply(
    lambda x: "yes" if x in matched_lab_names else "no")

In [ ]:


match_icu_freq_plot_perc = labs_icu_type_freq_perc[labs_icu_type_freq_perc["match"] == "yes"].hvplot.bar(x="labname",
                                                                                                         y="proportion",
                                                                                                         by="unittype",
                                                                                                         stacked=True,
                                                                                                         label="match").opts(
    xrotation=60, responsive=True, min_height=720, min_width=960)

In [ ]:
no_match_icu_freq_plot_perc = labs_icu_type_freq_perc[labs_icu_type_freq_perc["match"] == "no"].hvplot.bar(x="labname",
                                                                                                           y="proportion",
                                                                                                           by="unittype",
                                                                                                           stacked=True,
                                                                                                           label="no match").opts(
    xrotation=60, responsive=True, min_height=720, min_width=960)
labs_icu_plot_perc = match_icu_freq_plot_perc + no_match_icu_freq_plot_perc

labs_icu_plot_perc.opts(shared_axes=False, fontscale=1, title="All Lab Test Counts per ICU Type", )


### Lab Test Contribution of each ICU type 

In [ ]:
lab_test_contribution_total = df_labs_icu_type["unittype"].value_counts(normalize=False).reset_index()
lab_test_contribution_total_perc = df_labs_icu_type["unittype"].value_counts(normalize=True).reset_index()


In [ ]:
%matplotlib inline
make_donut_plot(data_values=lab_test_contribution_total["count"].to_list(),
                data_percentages=lab_test_contribution_total_perc["proportion"],
                data_labels=lab_test_contribution_total["unittype"].to_list(),
                plot_title="Lab Test Contributions across ICU Types", use_eng_fmt=True, use_value_fmt=True)

In [ ]:
df_labs_icu_type.head()

In [ ]:
df_match_or_no_match = pd.DataFrame(df_labs["labname"].unique().to_list())
df_match_or_no_match.columns = ["labname"]
df_match_or_no_match["match"] = df_match_or_no_match["labname"].map(lambda x: "yes" if x in matched_lab_names else "no")
pl_df_match_or_no_match = pl.from_pandas(df_match_or_no_match)

In [ ]:
pl_df_patients_icu_type.head()

In [ ]:
pl_df_labs_icu_type = pl_df_labs_icu_type.with_columns(pl.col("labname").cast(pl.String))

In [ ]:
pl_df_labs_icu_type_match = pl_df_labs_icu_type.join(pl_df_match_or_no_match, on="labname", how="left")
pl_df_labs_icu_type_match.head(n=10)
df_labs_icu_type_match: pd.DataFrame = pl_df_labs_icu_type_match.to_pandas()

In [ ]:
labs_icu_type_match_perc = df_labs_icu_type_match.value_counts(subset=["unittype", "match"],
                                                               normalize=True).reset_index()
labs_icu_type_match = df_labs_icu_type_match.value_counts(subset=["unittype", "match"], normalize=False).reset_index()

In [ ]:
labs_icu_type_match

In [ ]:
nested_donut_chart_data = {}

for icu_type in labs_icu_type_match["unittype"].unique():
    condition_1 = labs_icu_type_match.unittype == icu_type
    condition_2 = labs_icu_type_match.match == "yes"

    val_1 = labs_icu_type_match.loc[condition_1 & condition_2, 'count'].values[0]
    val_2 = labs_icu_type_match.loc[condition_1 & ~condition_2, 'count'].values[0]
    value_array = np.array([val_1, val_2])

    nested_donut_chart_data[icu_type] = value_array



In [ ]:
%matplotlib inline
fig_donut, ax_donut = plt.subplots(figsize=(10, 6), subplot_kw=dict(aspect="equal"))
wedge_data = np.array([value for value in nested_donut_chart_data.values()])

color_palette = sns.color_palette("inferno", 9)
color_palette_inner = sns.color_palette("inferno", 18)
color_palette_cb = palettable.tableau.ColorBlind_10.mpl_colors

hatches: list = ['/', '\\', '|', '-', '+', 'x', 'o', 'O', '.', '*', '/o', '\\|', '|*', '-\\', '+o', 'x*', 'o-', 'O|',
                 'O.', '*-']

size = 0.3
wedges_outer, texts = ax_donut.pie(wedge_data.sum(axis=1), wedgeprops=dict(width=0.3), startangle=-40,
                                   colors=color_palette)

inner_labels = (["yes", "no"] * 8)

wedges_inner = ax_donut.pie(wedge_data.flatten(), radius=1 - size, startangle=-40,
                            autopct="%1.1f%%", pctdistance=1.2,
                            colors=color_palette_inner, textprops={'color': "w"},
                            wedgeprops=dict(width=size + 0.2, edgecolor='w'))

bbox_props = dict(boxstyle="square,pad=0.3", fc="w", ec="k", lw=1)

# kw = dict(bbox=bbox_props, zorder=0, va="center")
kw = dict()

for i, p in enumerate(wedges_outer):
    ang = (p.theta2 - p.theta1) / 2. + p.theta1
    y = np.sin(np.deg2rad(ang))
    x = np.cos(np.deg2rad(ang))

    horizontalalignment = {-1: "right", 1: "left"}[int(np.sign(x))]

    x_text_mod = 1.2
    y_text_mod = 1.2

    x_text = x_text_mod * float(np.sign(x))
    y_text = y_text_mod * y

    label = lab_test_contribution_total_perc["unittype"][i]
    value_raw = lab_test_contribution_total_perc["proportion"][i]
    value = f"{value_raw:1.2%}"

    # var arrows
    ax_donut.annotate(f"{label} ({value})", xy=(x, y), xytext=(x_text, y_text),
                      horizontalalignment=horizontalalignment,
                      annotation_clip=False,
                      size=15,
                      arrowprops=dict(
                          arrowstyle="->",
                          facecolor="k",
                          edgecolor="k",
                          lw=2,
                          connectionstyle=f"angle,angleA=0,angleB={ang}"),
                      **kw)

plt.title("Lab Test Contributions of the available ICU Types", size=18)
plt.tight_layout()
plt.show()
